In [1]:
import pandas as pd

df = pd.read_csv('../dataset/email_spam_indo.csv')
df.head()

,Kategori,Pesan
0,spam,Secara alami tak tertahankan identitas perusah...
1,spam,Fanny Gunslinger Perdagangan Saham adalah Merr...
2,spam,Rumah -rumah baru yang luar biasa menjadi muda...
3,spam,4 Permintaan Khusus Pencetakan Warna Informasi...
4,spam,"Jangan punya uang, dapatkan CD perangkat lunak..."


In [2]:
df = df.rename(columns={
    'Pesan': 'text',
    'Kategori': 'label'
})

In [3]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(dikeret oleh|ditahan oleh|ect pada|subjek:).*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\w+\s+(com|net|org|co|id)\b', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'^(re|fw|fwd):[^.?!\n]*', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head()

,text,clean_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...


In [4]:
NORMALIZATION_DICT = {
    # 🔤 Singkatan umum
    "gk": "tidak",
    "d": "di",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",
    "dr": "dari",
    "dgn": "dengan",
    "utk": "untuk",
    "yg": "yang",
    "jg": "juga",
    "krn": "karena",
    "blm": "belum",
    "sdh": "sudah",
    "aja": "saja",
    "trs": "terus",
    "bgt": "banget",
    "dlm": "dalam",

    # 💰 Kata khas spam / promo
    "rp": "rupiah",
    "jt": "juta",
    "rb": "ribu",
    "cashback": "cashback",
    "promo": "promosi",
    "diskon": "diskon",
    "gratis": "gratis",
    "hadiah": "hadiah",
    "menang": "menang",
    "undian": "undian",

    # 📱 Variasi kata penting
    "tlp": "telepon",
    "telp": "telepon",
    "hp": "handphone",
    "no": "nomor",
    "rek": "rekening",
    "an": "atas nama",

    # ⚡ Variasi penulisan spam
    "selamat!!!": "selamat",
    "selamat!!": "selamat",
    "selamat!": "selamat",
    "trnsfer": "transfer",
    "trf": "transfer",

    # 🧩 Variasi angka ke huruf (leetspeak)
    "4nda": "anda",
    "4pa": "apa",
    "1ni": "ini",
    "k4mu": "kamu",

    # 🪤 Kata clickbait spam
    "klik": "klik",
    "segera": "segera",
    "buruan": "cepat",
    "cepat": "cepat",
    "terbatas": "terbatas",
    "khusus": "khusus",

    # 🎯 Kata inti spam
    "free": "gratis",
    "win": "menang",
    "winner": "pemenang",
    "prize": "hadiah",
    "bonus": "bonus",
    "gift": "hadiah",
    "reward": "hadiah",

    # 💰 Finansial / uang
    "cash": "uang",
    "money": "uang",
    "credit": "kredit",
    "loan": "pinjaman",
    "bank": "bank",
    "transfer": "transfer",
    "account": "akun",
    "sspecial": "spesial",
    "balance": "saldo",

    # 📢 Promosi / marketing
    "promo": "promosi",
    "offer": "penawaran",
    "deal": "penawaran",
    "discount": "diskon",
    "sale": "diskon",
    "limited": "terbatas",
    "exclusive": "eksklusif",

    # ⚡ Call to action (penting banget buat spam)
    "click": "klik",
    "claim": "klaim",
    "buy": "beli",
    "order": "pesan",
    "register": "daftar",
    "subscribe": "langganan",
    "join": "gabung",
    "verify": "verifikasi",
    "confirm": "konfirmasi",

    # ⏰ urgency
    "urgent": "segera",
    "now": "sekarang",
    "today": "hari ini",
    "instant": "instan",

    # 🖥️ teknologi / produk
    "software": "perangkat lunak",
    "system": "sistem",
    "update": "pembaruan",
    "download": "unduh",
    "install": "instal",

    # 📧 email umum
    "hello": "halo",
    "dear": "halo",
    "sir": "bapak",
    "madam": "ibu",

    # 🧩 lain-lain yang sering muncul
    "service": "layanan",
    "customer": "pelanggan",
    "support": "dukungan",
    "information": "informasi",
    "message": "pesan"
}
def normalize_text(text):
    words = text.split()
    normalized_words = [NORMALIZATION_DICT.get(word, word) for word in words]
    return ' '.join(normalized_words)

df['normalized_text'] = df['clean_text'].apply(normalize_text)
df[['text', 'clean_text','normalized_text']].head()


,text,clean_text,normalized_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [5]:
!pip install Sastrawi
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ambil stopword default
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

# fungsi hapus stopword
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopwords])

# apply ke data
df['stopwords'] = df['normalized_text'].apply(remove_stopwords)

# lihat hasil
df[['text', 'normalized_text', 'stopwords']].head()

,text,normalized_text,stopwords
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [7]:
# tokenization
df['tokens'] = df['stopwords'].apply(lambda x: x.split())

# lihat hasil
df[['text', 'clean_text', 'stopwords', 'tokens']].head()

,text,clean_text,stopwords,tokens
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...,"[alami, tak, tertahankan, identitas, perusahaa..."
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...,"[fanny, gunslinger, perdagangan, saham, merril..."
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...,"[rumah, rumah, baru, luar, biasa, menjadi, mud..."
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...,"[permintaan, khusus, pencetakan, warna, inform..."
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...,"[jangan, punya, uang, dapatkan, cd, perangkat,..."


In [8]:
!pip install pycaret scikit-learn gensim


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# Model Word2Vec butuh data berformat list kata per kalimat (ada di df['tokens'])
# Kita buat 100 fitur/dimensi vektor untuk setiap kalimat
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

# Fungsi pembuat Sentence/Document Vector (Rata-rata fitur vektor dari kumpulan kata)
def document_vector(doc):
    valid_words = [word for word in doc if word in w2v_model.wv.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(100)
    return np.mean(w2v_model.wv[valid_words], axis=0)

X_w2v = np.array(df['tokens'].apply(document_vector).tolist())

# Susun ke bentuk DataFrame untuk PyCaret
feature_names = [f"w2v_{i}" for i in range(100)]
df_pycaret = pd.DataFrame(X_w2v, columns=feature_names)
df_pycaret['label'] = df['label'].values

df_pycaret.head()

,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99,label
0,0.125540,0.789773,0.497042,0.023855,-0.163746,-0.503382,0.185292,0.623011,-0.232935,-0.675609,...,0.134053,0.101410,0.380903,0.925785,0.207960,0.856033,-0.551325,0.481667,-0.416497,spam
1,0.024093,0.368685,0.349106,0.005140,-0.064281,-0.370475,0.166030,0.511168,-0.260130,-0.408631,...,0.051776,0.113358,0.153400,0.553900,0.145952,0.334525,-0.444082,0.412654,-0.195585,spam
2,0.150292,0.614925,0.501661,-0.012514,-0.126679,-0.548398,0.172184,0.737906,-0.332125,-0.644213,...,0.104734,0.124555,0.434004,0.885342,0.215604,0.695138,-0.639739,0.514008,-0.422206,spam
3,0.156295,0.304377,0.057753,-0.191023,0.552712,-1.023975,-0.289323,1.493322,-0.127276,-0.011045,...,0.250586,0.617350,0.665449,1.078690,0.396868,0.148240,-0.344388,0.031836,-0.322384,spam
4,0.157275,0.514974,0.299247,0.044159,-0.140922,-0.239543,0.333316,0.840438,-0.706737,-0.214141,...,0.317189,0.264564,0.468735,1.127959,0.343175,0.450384,-0.165402,0.548892,-0.762411,spam


In [10]:
from pycaret.classification import ClassificationExperiment

# Inisialisasi Environment PyCaret berbasis Object Khusus untuk W2V
exp_w2v = ClassificationExperiment()
clf_setup = exp_w2v.setup(data=df_pycaret, target='label', session_id=42, use_gpu=False)

,Description,Value
0,Session id,42
1,Target,label
2,Target type,Binary
3,Target mapping,"ham: 0, spam: 1"
4,Original data shape,"(2636, 101)"
5,Transformed data shape,"(2636, 101)"
6,Transformed train set shape,"(1845, 101)"
7,Transformed test set shape,"(791, 101)"
8,Numeric features,100
9,Preprocess,True


In [11]:
# Membandingkan model SVM, Naive Bayes (nb), dan Logistic Regression (lr)
best_model = exp_w2v.compare_models(include=['svm', 'nb', 'lr'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
svm,SVM - Linear Kernel,0.9019,0.9719,0.9019,0.9092,0.9015,0.8039,0.8109,1.2060
lr,Logistic Regression,0.8959,0.9588,0.8959,0.8974,0.8957,0.7911,0.7928,0.0330
nb,Naive Bayes,0.7518,0.8773,0.7518,0.7893,0.7411,0.4960,0.5352,0.7870


In [12]:
# --- EKSPOR SELURUH MODEL (SVM, NB, LR) & W2V KAMUS ---
print("Memulai kompilasi SVM...")
svm_model = exp_w2v.create_model('svm')
exp_w2v.save_model(svm_model, 'spam_model_svm')

print("Memulai kompilasi Naive Bayes...")
nb_model = exp_w2v.create_model('nb')
exp_w2v.save_model(nb_model, 'spam_model_nb')

print("Memulai kompilasi Logistic Regression...")
lr_model = exp_w2v.create_model('lr')
exp_w2v.save_model(lr_model, 'spam_model_lr')

print("Memulai kompilasi Engine Text Word2Vec...")
w2v_model.save('w2v_kamus.model')

print("STATUS: BERHASIL! Segala Pkl dan Model Bin telah disimpan dalam folder!")

Memulai kompilasi SVM...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9568,0.9782,0.9568,0.9575,0.9567,0.9132,0.9141
1,0.9027,0.9738,0.9027,0.9054,0.9024,0.8045,0.8075
2,0.8757,0.9772,0.8757,0.8895,0.8751,0.7527,0.7656
3,0.8919,0.9827,0.8919,0.9046,0.8914,0.7849,0.7968
4,0.8811,0.9759,0.8811,0.8968,0.8804,0.7636,0.7783
5,0.8750,0.9711,0.8750,0.8833,0.8748,0.7511,0.7586
6,0.8913,0.9653,0.8913,0.8939,0.8909,0.7814,0.7844
7,0.8859,0.9406,0.8859,0.8864,0.8859,0.7717,0.7721
8,0.9022,0.9597,0.9022,0.9178,0.9009,0.8029,0.8189


Transformation Pipeline and Model Successfully Saved
Memulai kompilasi Naive Bayes...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7730,0.8894,0.7730,0.7846,0.7693,0.5415,0.5549
1,0.7568,0.8752,0.7568,0.7867,0.7480,0.5065,0.5386
2,0.7568,0.8902,0.7568,0.8030,0.7445,0.5053,0.5531
3,0.7459,0.9057,0.7459,0.8111,0.7287,0.4820,0.5474
4,0.7405,0.8672,0.7405,0.7703,0.7306,0.4734,0.5056
5,0.7446,0.8678,0.7446,0.7647,0.7372,0.4815,0.5044
6,0.7663,0.8846,0.7663,0.8220,0.7528,0.5229,0.5800
7,0.7337,0.8543,0.7337,0.7705,0.7218,0.4601,0.4987
8,0.7609,0.8962,0.7609,0.8058,0.7496,0.5150,0.5611


Transformation Pipeline and Model Successfully Saved
Memulai kompilasi Logistic Regression...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9351,0.9695,0.9351,0.9353,0.9351,0.8700,0.8702
1,0.8811,0.9560,0.8811,0.8811,0.8811,0.7618,0.7618
2,0.9189,0.9641,0.9189,0.9193,0.9188,0.8374,0.8378
3,0.9135,0.9711,0.9135,0.9164,0.9132,0.8262,0.8293
4,0.8865,0.9613,0.8865,0.8884,0.8862,0.7720,0.7742
5,0.8859,0.9638,0.8859,0.8860,0.8859,0.7714,0.7715
6,0.8641,0.9451,0.8641,0.8658,0.8637,0.7268,0.7290
7,0.8641,0.9284,0.8641,0.8641,0.8641,0.7279,0.7279
8,0.8859,0.9475,0.8859,0.8911,0.8852,0.7706,0.7762


Transformation Pipeline and Model Successfully Saved
Memulai kompilasi Engine Text Word2Vec...
STATUS: BERHASIL! Segala Pkl dan Model Bin telah disimpan dalam folder!
